# 01b — Bronze Backfill: Ingest a Date Range from TED

**Purpose**

One-time backfill notebook. Loops over a range of past business days, downloads each day's TED package, extracts the XML to Bronze, and deletes the `.tar.gz` to save space. After running this once, you go back to `01_ingest_xml` for the daily fetch.

**Default range:** 2026-05-18 → 2026-06-17 (one month ending the day before today). Change the widgets in Cell 2 if you need a different window.

**Storage notes**

- ~22 publication days in a month (TED skips weekends and EU holidays).
- ~140 MB of XML per day → roughly 3 GB across the month.
- The `.tar.gz` is deleted after extraction (Cell 5). The Silver notebook reads the XML, not the archive — losing the archive is safe.
- The notebook **skips dates that already exist in Bronze**, so re-running it is cheap and idempotent.

## Cell 1 — Imports

In [0]:
# ── CELL 1 — Imports ─────────────────────────────────────────────────────────
import os
import csv
import urllib.request
import tarfile
from datetime import datetime, date, timedelta
from io import StringIO

## Cell 2 — Date-range widgets

In [0]:
# ── CELL 2 — Date-range widgets ──────────────────────────────────────────────
dbutils.widgets.text("start_date", "2026-05-18", "Backfill start YYYY-MM-DD")
dbutils.widgets.text("end_date",   "2026-06-17", "Backfill end   YYYY-MM-DD")

start_date = datetime.strptime(dbutils.widgets.get("start_date"), "%Y-%m-%d").date()
end_date   = datetime.strptime(dbutils.widgets.get("end_date"),   "%Y-%m-%d").date()

VOLUME_ROOT = "/Volumes/workspace/default/lakehouse/bronze/ted"

print(f"Backfill range : {start_date} → {end_date}")
print(f"Bronze root    : {VOLUME_ROOT}")

## Cell 3 — Publication-number lookup

Same logic as Notebook 01. Maps a calendar date to TED's OJS publication number (e.g. 2026-06-12 → `202600112`).

In [0]:
# ── CELL 3 — Publication-number lookup (from local CSV) ──────────────────────
# Why local instead of HTTP: TED's release-calendar endpoint sits behind AWS WAF
# and challenges non-browser clients (returns 202 + empty body forever). Easier
# and more reliable to download the CSV once per year from a browser and read it
# from a Unity Catalog Volume.
#
# Expected file: /Volumes/workspace/default/lakehouse/bronze/calendar/ted_calendar_YYYY.csv
# Format (as TED serves it):
#     OJS,Publication date
#     1,02/01/2026
#     2,05/01/2026
#     ...

CALENDAR_ROOT = "/Volumes/workspace/default/lakehouse/bronze/calendar"

_calendar_cache = {}  # year -> {date: ojs_string}

def _parse_csv_date(s):
    """Try common formats; return a date or None."""
    s = s.strip()
    for fmt in ("%d/%m/%Y", "%Y-%m-%d", "%d.%m.%Y", "%Y/%m/%d"):
        try:
            return datetime.strptime(s, fmt).date()
        except ValueError:
            continue
    return None

def _load_year_calendar(year):
    """Read the local TED calendar CSV for `year` and return {date: ojs_string}."""
    if year in _calendar_cache:
        return _calendar_cache[year]

    path = f"{CALENDAR_ROOT}/ted_calendar_{year}.csv"
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing {path}. Download it in your browser from "
            f"https://ted.europa.eu/en/release-calendar/-/download/file/CSV/{year} "
            f"and upload to that Volume path."
        )

    mapping = {}
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        headers = [h.strip().lower() for h in next(reader, [])]
        # Find which column is the date and which is the OJS number.
        date_idx, ojs_idx = 1, 0  # TED default: OJS first, date second
        for i, h in enumerate(headers):
            if "date" in h:
                date_idx = i
            if "ojs" in h or "issue" in h or "number" in h:
                ojs_idx = i
        for row in reader:
            if len(row) <= max(date_idx, ojs_idx):
                continue
            d = _parse_csv_date(row[date_idx])
            digits = "".join(c for c in row[ojs_idx] if c.isdigit())
            if d and digits:
                mapping[d] = f"{year}{int(digits):05d}"

    _calendar_cache[year] = mapping
    print(f"  loaded {year} calendar from {path}: {len(mapping)} publication days")
    return mapping

def get_publication_number(target_date):
    """Return TED publication ID (e.g. '202600112'), or None if not a publishing day."""
    return _load_year_calendar(target_date.strftime("%Y")).get(target_date)

## Cell 4 — Backfill loop

For each business day in the range:

1. Skip if the day's folder already exists in Bronze with XML in it (idempotent re-run).
2. Look up the publication number.
3. Download the archive.
4. Extract the XML.
5. Delete the archive to save space.

Non-publishing days (weekends, holidays) fail at step 2 and are skipped quietly.

In [0]:
# ── CELL 4 — Backfill loop ───────────────────────────────────────────────────
def folder_has_xml(folder):
    """True if the folder exists and contains at least one .xml file."""
    if not os.path.exists(folder):
        return False
    for _, _, files in os.walk(folder):
        for f in files:
            if f.endswith(".xml"):
                return True
    return False

def ingest_one_day(day):
    YYYY     = day.strftime("%Y")
    MM       = day.strftime("%m")
    YYYYMMDD = day.strftime("%Y%m%d")
    folder   = f"{VOLUME_ROOT}/{YYYY}/{MM}/{YYYYMMDD}"

    if folder_has_xml(folder):
        return ("SKIP", "already in Bronze")

    pub = get_publication_number(day)
    if not pub:
        return ("SKIP", "not a publishing day")

    dbutils.fs.mkdirs(folder)
    archive = f"{folder}/{pub}.tar.gz"

    # Download
    urllib.request.urlretrieve(f"https://ted.europa.eu/packages/daily/{pub}", archive)

    # Extract
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(folder)

    # Delete archive — the XML is on disk, the .tar.gz is redundant and costs ~18 MB per day.
    os.remove(archive)

    # Count what we just wrote
    xml_count = 0
    for _, _, files in os.walk(folder):
        xml_count += sum(1 for f in files if f.endswith(".xml"))
    return ("OK", f"{xml_count:,} XML files")

# Iterate inclusively from start_date to end_date.
results = []
day = start_date
while day <= end_date:
    try:
        status, detail = ingest_one_day(day)
    except Exception as e:
        status, detail = ("ERR", str(e)[:120])
    print(f"{day}  {status:4s}  {detail}")
    results.append((day, status, detail))
    day += timedelta(days=1)

## Cell 5 — Verify what Bronze now contains

In [0]:
# ── CELL 5 — Verify Bronze contents ──────────────────────────────────────────
# Walk the Bronze tree and count XML files per date folder.
print("Files per date folder in Bronze:")
total = 0
for year in sorted(os.listdir(VOLUME_ROOT)):
    year_path = os.path.join(VOLUME_ROOT, year)
    if not os.path.isdir(year_path):
        continue
    for month in sorted(os.listdir(year_path)):
        month_path = os.path.join(year_path, month)
        if not os.path.isdir(month_path):
            continue
        for day_folder in sorted(os.listdir(month_path)):
            day_path = os.path.join(month_path, day_folder)
            if not os.path.isdir(day_path):
                continue
            xml_count = 0
            for _, _, files in os.walk(day_path):
                xml_count += sum(1 for f in files if f.endswith(".xml"))
            total += xml_count
            print(f"  {day_folder}: {xml_count:,} XML files")

print(f"\nTotal XML files in Bronze: {total:,}")

In [0]:
_calendar_cache.clear()

## Next step

Once this finishes (expect 5–10 minutes per day across the network), run **`02b_parse_notices_backfill`** to push the full month into Silver in one shot. After that, go back to running `02_parse_notices` daily.